# Week 39, Day 1: Hearing: the cochlea as a filter bank

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w039/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Filters spaced by ERB

Write erb(f), and cfs_between(lo, hi, n): n centre frequencies equally spaced on the ERB-number scale between lo and hi. (Convert lo and hi to ERB numbers, space them linearly, and convert back.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fs = 16000

def erb(f):
    return ...

def erb_number(f):
    return 21.4 * np.log10(4.37 * f / 1000 + 1)

def cfs_between(lo, hi, n):
    e = np.linspace(erb_number(lo), erb_number(hi), n)
    return ...

cfs = cfs_between(100, 6000, 32)
print("first centre frequencies:", np.round(cfs[:5]), "  last:", np.round(cfs[-3:]))
plt.semilogy(cfs, "o"); plt.ylabel("centre frequency (Hz)"); plt.xlabel("filter number"); plt.show()

In [ ]:
import numpy as np
def test_erb():
    "ERB is about 133 Hz at 1 kHz"
    assert abs(erb(1000) - 132.6) < 1

def test_spacing():
    "neighbouring centre frequencies are about one ERB apart, and the spacing grows with frequency"
    d = np.diff(cfs)
    assert np.all(np.diff(d) > 0) and abs(d[len(d) // 2] / erb(cfs[len(d) // 2]) - 1) < 0.1

test_erb()
test_spacing()
print("All checks passed")

### Exercise: A gammatone filter and its bandwidth

Write gammatone(fc, dur=0.05), the order-4 gammatone impulse response sampled at fs (normalised to unit peak). Then measure its equivalent rectangular bandwidth: the area under the power response |H(f)|² divided by its peak value.

In [ ]:
def gammatone(fc, dur=0.05):
    t = np.arange(int(dur * fs)) / fs
    b = 1.019 * erb(fc)
    g = ...
    return g / np.abs(g).max()

g = gammatone(1000)
H = np.fft.rfft(g, 1 << 15); f = np.fft.rfftfreq(1 << 15, 1 / fs)
P = np.abs(H) ** 2
peak_f = f[np.argmax(P)]
erb_measured = ...
print(f"peak at {peak_f:.0f} Hz; measured ERB {erb_measured:.1f} Hz against formula {erb(1000):.1f} Hz")
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(np.arange(len(g)) / fs * 1000, g); ax[0].set_xlabel("ms")
ax[1].plot(f, 10 * np.log10(P / P.max())); ax[1].set_xlim(0, 3000); ax[1].set_ylim(-40, 2); ax[1].set_xlabel("Hz"); plt.show()

In [ ]:
import numpy as np
def test_cf():
    "the filter's frequency response peaks within 2% of its centre frequency"
    assert abs(peak_f - 1000) / 1000 < 0.02

def test_erb_bw():
    "its measured equivalent rectangular bandwidth is within 10% of ERB(1000)"
    assert abs(erb_measured / erb(1000) - 1) < 0.1

test_cf()
test_erb_bw()
print("All checks passed")

### Exercise: The excitation pattern of a harmonic complex

For each harmonic h = 1 to 20 of 150 Hz, call it resolved if the excitation pattern (evaluated with a dense filter bank) has a local peak within a quarter of F0 of h times F0 that is at least 2 dB above the dip halfway to the next harmonic. The filtering is done for you.

In [ ]:
F0, dur = 150.0, 0.2
t = np.arange(int(dur * fs)) / fs
sound = sum(np.sin(2 * np.pi * h * F0 * t) for h in range(1, 31))
dense = cfs_between(80, 4000, 240)
level = np.array([np.sqrt(np.mean(np.convolve(sound, gammatone(fc), mode="same")[800:] ** 2)) for fc in dense])
level_db = 20 * np.log10(level / level.max())

def db_at(freq):
    return np.interp(freq, dense, level_db)

resolved = {}
for h in range(1, 21):
    peak = ...
    dip = ...
    resolved[h] = ...
print("resolved harmonics:", [h for h, v in resolved.items() if v])
plt.semilogx(dense, level_db); [plt.axvline(h * F0, color="gray", lw=0.4) for h in range(1, 21)]
plt.xlabel("filter centre frequency (Hz)"); plt.ylabel("excitation (dB)"); plt.show()

In [ ]:
import numpy as np
def test_low():
    "the 2nd and 3rd harmonics make separate peaks"
    assert resolved[2] and resolved[3]

def test_high():
    "the 12th harmonic is not resolved, and between 5 and 11 harmonics are resolved in all"
    assert not resolved[12] and 5 <= sum(resolved.values()) <= 11

test_low()
test_high()
print("All checks passed")